# Task 3: End-to-End ABSA with PhoBERT
## Unified Model: Extraction + Classification in ONE model

PhoBERT-CRF hoc truc tiep Unified Tags (B-CAMERA#POSITIVE, ...).
So sanh voi Pipeline (Notebook 03) de ket luan phuong phap nao tot hon.


## 1. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import load_raw_data
from src.e2e.e2e_dataset import E2EDataset, BIO_TAGS, TAG2ID, NUM_TAGS, LABEL_NAMES
from src.e2e.e2e_model import E2EPhoBertCRF
from src.utils.engine import train_e2e_model, predict_e2e
from src.utils.metrics import bio_tags_to_spans, evaluate_spans_f1, token_accuracy
from src.utils.visualization import plot_training_curves, plot_model_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)
print(f"Device: {device}")


## 2. Load Data (PhoBERT co tokenizer rieng, khong can segment)


In [ ]:
DATA_DIR = os.path.join("..", "..", "data")

train_items = load_raw_data(os.path.join(DATA_DIR, "train.jsonl"))
dev_items = load_raw_data(os.path.join(DATA_DIR, "dev.jsonl"))
test_items = load_raw_data(os.path.join(DATA_DIR, "test.jsonl"))
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")


## 3. E2E Dataset (PhoBERT Tokenizer + Subword Alignment)


In [ ]:
MAX_LEN = 256
BATCH_SIZE = 16  # PhoBERT rat nang, phai dung batch nho

print("Tokenizing with PhoBERT (may take a few minutes)...")
train_ds = E2EDataset(train_items, max_len=MAX_LEN)
dev_ds = E2EDataset(dev_items, max_len=MAX_LEN)
test_ds = E2EDataset(test_items, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)
print(f"Ready! Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")


## 4. Train PhoBERT-CRF (Grid Search LR)


In [ ]:
LR_OPTIONS = [2e-5, 5e-5]

all_models = {}
all_histories = {}
all_results = []

for lr in LR_OPTIONS:
    name = f"PhoBERT-CRF_lr{lr}"
    model = E2EPhoBertCRF(num_unified_tags=NUM_TAGS, dropout=0.3).to(device)
    model, history = train_e2e_model(
        model, train_loader, dev_loader, device,
        lr=lr, epochs=15, patience=5, model_name=name
    )
    all_models[name] = model
    all_histories[name] = history


## 5. Evaluate on Test Set


In [ ]:
for name, model in all_models.items():
    test_res = predict_e2e(model, test_loader, device)

    pred_spans = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res["pred_tags"], test_res["lengths"])]
    true_spans = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res["true_tags"], test_res["lengths"])]
    span_f1 = evaluate_spans_f1(pred_spans, true_spans)
    tok_acc = token_accuracy(test_res["pred_tags"], test_res["true_tags"])

    all_results.append({
        "Model": name,
        "Span_P": round(span_f1["precision"], 4),
        "Span_R": round(span_f1["recall"], 4),
        "Span_F1": round(span_f1["f1"], 4),
        "Token_Acc": round(tok_acc, 4),
    })
    print(f"{name} | Span F1: {span_f1['f1']:.4f} | TokAcc: {tok_acc:.4f}")

e2e_df = pd.DataFrame(all_results)
display(e2e_df.style.highlight_max(subset=["Span_F1"], color="lightgreen"))


## 6. So Sanh Pipeline vs E2E


In [ ]:
print("\n" + "="*60)
print("  PIPELINE vs END-TO-END COMPARISON")
print("="*60)

# Load pipeline result
try:
    pipeline_f1 = pd.read_csv("../../results/pipeline/error_analysis.csv")
    # Recalculate pipeline F1 from the error counts
    # (or load from saved metrics)
    print("Pipeline results loaded from Notebook 03")
except:
    print("(Chua co ket qua Pipeline. Hay chay Notebook 03 truoc.)")

# Show E2E results
print("\nE2E Results:")
display(e2e_df)

# Load ATE-only results for reference
try:
    ate_df = pd.read_csv("../../results/ate/ate_results.csv")
    print("\nATE-only Baselines (for reference):")
    display(ate_df)
except:
    pass


## 7. Error Analysis


In [ ]:
best_name = e2e_df.loc[e2e_df["Span_F1"].idxmax(), "Model"]
best_model = all_models[best_name]
best_test = predict_e2e(best_model, test_loader, device)

errors = []
for i in range(len(test_items)):
    true_labels = set(label for _, _, label in test_items[i].get("labels", []))
    pred_spans = bio_tags_to_spans(best_test["pred_tags"][i], BIO_TAGS, best_test["lengths"][i])
    pred_labels = set(span[0] for span in pred_spans)

    fp = pred_labels - true_labels
    fn = true_labels - pred_labels

    if fp or fn:
        errors.append({
            "Text": test_items[i]["text"][:100],
            "True": ", ".join(sorted(true_labels)),
            "Pred": ", ".join(sorted(pred_labels)),
            "FP": len(fp), "FN": len(fn),
        })

error_df = pd.DataFrame(errors)
print(f"Cau co loi: {len(error_df)} / {len(test_items)} ({len(error_df)/len(test_items)*100:.1f}%)")
display(error_df.head(15))


## 8. Save


In [ ]:
SAVE_DIR = os.path.join("..", "..", "results", "e2e")
os.makedirs(SAVE_DIR, exist_ok=True)
e2e_df.to_csv(os.path.join(SAVE_DIR, "e2e_results.csv"), index=False)
torch.save(best_model.state_dict(), os.path.join(SAVE_DIR, "best_e2e_phobert.pt"))
print(f"Best E2E: {best_name} -> Saved!")
